In [5]:
# NẠP DỮ LIỆU ĐẦU VÀO
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name in {"notebooks", "src"} else current_dir

features_path = project_root / "data" / "processed" / "cafeland_features.csv"
df_features = pd.read_csv(features_path)
print(f"Đã nạp {df_features.shape[0]:,} dòng x {df_features.shape[1]} cột từ {features_path.name}")
df_features.dtypes

Đã nạp 5,261 dòng x 20 cột từ cafeland_features.csv


tieu_de                                    str
gia                                    float64
dien_tich                              float64
vi_tri                                     str
ngay_dang                                  str
so_phong_ngu                           float64
price_per_m2                           float64
khu_vuc_nhom                               str
loai_bds                                   str
gia_thieu                                 bool
dien_tich_thieu                           bool
so_phong_ngu_thieu                        bool
kv_Bà Rịa - Vũng Tàu (mới sáp nhập)       bool
kv_Bình Dương (mới sáp nhập)              bool
kv_TP.HCM (lõi cũ)                        bool
loai_Biệt thự                             bool
loai_Căn hộ/Chung cư                      bool
loai_Khác                                 bool
loai_Nhà ở/Nhà phố                        bool
loai_Đất nền                              bool
dtype: object

In [6]:
# lOẠI BỎ CÁC CỘT ĐỊNH TÍNH
#  - tieu_de, vi_tri, ngay_dang: là văn bản hay thời gian tự do, không phải đặc trưng số
#  - khu_vuc_nhom, loai_bds: đã có bản one-hot (kv_..., loai_...) thay thế
drop_cols = ["tieu_de", "vi_tri", "ngay_dang", "khu_vuc_nhom", "loai_bds"]
df_model = df_features.drop(columns=[c for c in drop_cols if c in df_features.columns])

print(f"Còn lại {df_model.shape[1]} cột:")
print(list(df_model.columns))
df_model.head()

Còn lại 15 cột:
['gia', 'dien_tich', 'so_phong_ngu', 'price_per_m2', 'gia_thieu', 'dien_tich_thieu', 'so_phong_ngu_thieu', 'kv_Bà Rịa - Vũng Tàu (mới sáp nhập)', 'kv_Bình Dương (mới sáp nhập)', 'kv_TP.HCM (lõi cũ)', 'loai_Biệt thự', 'loai_Căn hộ/Chung cư', 'loai_Khác', 'loai_Nhà ở/Nhà phố', 'loai_Đất nền']


,gia,dien_tich,so_phong_ngu,price_per_m2,gia_thieu,dien_tich_thieu,so_phong_ngu_thieu,kv_Bà Rịa - Vũng Tàu (mới sáp nhập),kv_Bình Dương (mới sáp nhập),kv_TP.HCM (lõi cũ),loai_Biệt thự,loai_Căn hộ/Chung cư,loai_Khác,loai_Nhà ở/Nhà phố,loai_Đất nền
0,2.990000e+09,72.0,2.0,4.152778e+07,False,False,False,False,False,True,False,False,True,False,False
1,5.350000e+09,100.0,3.0,5.350000e+07,False,False,False,False,False,True,False,False,False,True,False
2,3.750000e+09,105.0,NaN,3.571429e+07,False,False,True,False,False,True,False,False,False,False,True
3,4.490000e+09,100.0,2.0,4.490000e+07,False,False,False,False,False,True,False,False,False,True,False
4,7.900000e+09,120.0,NaN,6.583333e+07,False,False,True,False,False,True,False,False,False,True,False


In [7]:
# CHECK DỮ LIỆU TRƯỚC KHI LƯU
# Chỉ còn cột số hoặc boolean (one-hot)
non_numeric = df_model.select_dtypes(exclude=["number", "bool"]).columns.tolist()
assert not non_numeric, f"Vẫn còn cột không phải số: {non_numeric}"
assert "price_per_m2" in df_model.columns, "Thiếu biến mục tiêu price_per_m2"

print(f"\nShape cuối cùng: {df_model.shape}")
df_model.head()


Shape cuối cùng: (5261, 15)


,gia,dien_tich,so_phong_ngu,price_per_m2,gia_thieu,dien_tich_thieu,so_phong_ngu_thieu,kv_Bà Rịa - Vũng Tàu (mới sáp nhập),kv_Bình Dương (mới sáp nhập),kv_TP.HCM (lõi cũ),loai_Biệt thự,loai_Căn hộ/Chung cư,loai_Khác,loai_Nhà ở/Nhà phố,loai_Đất nền
0,2.990000e+09,72.0,2.0,4.152778e+07,False,False,False,False,False,True,False,False,True,False,False
1,5.350000e+09,100.0,3.0,5.350000e+07,False,False,False,False,False,True,False,False,False,True,False
2,3.750000e+09,105.0,NaN,3.571429e+07,False,False,True,False,False,True,False,False,False,False,True
3,4.490000e+09,100.0,2.0,4.490000e+07,False,False,False,False,False,True,False,False,False,True,False
4,7.900000e+09,120.0,NaN,6.583333e+07,False,False,True,False,False,True,False,False,False,True,False


In [8]:
# LƯU FILE
output_path = project_root / "data" / "processed" / "cleaned_for_model.csv"
df_model.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu {df_model.shape[0]:,} dòng x {df_model.shape[1]} cột -> {output_path}")

Đã lưu 5,261 dòng x 15 cột -> D:\iuh\hk1_nam3\DAP\Project\data\processed\cleaned_for_model.csv
